# TrafficIQ — Baseline Notebook

**Goal:** Predict traffic demand (0–1) from geographic, road, and weather features.

**Metric:** `score = max(0, 100 × R²)`

---
Steps covered here:
1. Load data
2. Explore & understand
3. Clean missing values
4. Feature engineering
5. Train a baseline model
6. Validate locally
7. Generate submission file

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print('Libraries loaded.')

## 1. Load Data

In [ ]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
train.head()

## 2. Explore

In [ ]:
train.info()

In [ ]:
# Missing value counts
missing = train.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
# Target distribution
plt.figure(figsize=(8, 3))
plt.hist(train['demand'], bins=50, color='steelblue', edgecolor='white')
plt.title('Demand Distribution')
plt.xlabel('demand')
plt.tight_layout()
plt.show()

print(train['demand'].describe())

In [ ]:
# Categorical value counts
for col in ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']:
    print(f'\n{col}:')
    print(train[col].value_counts(dropna=False))

## 3. Clean Missing Values

In [ ]:
def clean(df):
    df = df.copy()

    # Fill categoricals with mode (most frequent)
    for col in ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']:
        if col in df.columns:
            mode_val = df[col].mode()[0]
            df[col] = df[col].fillna(mode_val)

    # Fill Temperature with median
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())

    return df

train = clean(train)
test  = clean(test)

print('Missing values after cleaning:')
print(train.isnull().sum()[train.isnull().sum() > 0])
print('All clean!' if train.isnull().sum().sum() == 0 else 'Still some missing!')

## 4. Feature Engineering

In [ ]:
def engineer(df):
    df = df.copy()

    # Parse timestamp -> hour, minute, time_of_day (0–96 slots of 15 min)
    time_parts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour']        = time_parts[0]
    df['minute']      = time_parts[1]
    df['time_slot']   = df['hour'] * 4 + df['minute'] // 15  # 0–95

    # Cyclical encoding of hour (captures 23->0 continuity)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

    # Day-of-week proxy (day % 7)
    df['day_of_week'] = df['day'] % 7

    # Binary flags
    df['is_large_vehicles'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['has_landmark']      = (df['Landmarks'] == 'Yes').astype(int)

    # Encode categoricals
    le = LabelEncoder()
    for col in ['RoadType', 'Weather', 'geohash']:
        df[col + '_enc'] = le.fit_transform(df[col].astype(str))

    return df

train = engineer(train)
test  = engineer(test)

print('New columns added:')
new_cols = ['hour', 'minute', 'time_slot', 'hour_sin', 'hour_cos',
            'day_of_week', 'is_large_vehicles', 'has_landmark',
            'RoadType_enc', 'Weather_enc', 'geohash_enc']
print(train[new_cols].head())

## 5. Prepare Feature Matrix

In [ ]:
FEATURES = [
    'geohash_enc',
    'day',
    'day_of_week',
    'hour',
    'minute',
    'time_slot',
    'hour_sin',
    'hour_cos',
    'RoadType_enc',
    'NumberofLanes',
    'is_large_vehicles',
    'has_landmark',
    'Temperature',
    'Weather_enc',
]
TARGET = 'demand'

X = train[FEATURES]
y = train[TARGET]
X_test = test[FEATURES]

print(f'X shape:      {X.shape}')
print(f'X_test shape: {X_test.shape}')

## 6. Train — Random Forest Baseline

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'Train: {X_train.shape}, Val: {X_val.shape}')

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=SEED,
)
model.fit(X_train, y_train)
print('Training done.')

## 7. Validate Locally

In [ ]:
val_preds = model.predict(X_val)
r2 = r2_score(y_val, val_preds)
score = max(0, 100 * r2)

print(f'R²:    {r2:.4f}')
print(f'Score: {score:.2f}')
print()
print('Rule: only submit if this score is better than our current best.')

In [ ]:
# Feature importance
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(7, 5))
importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual scatter
plt.figure(figsize=(5, 5))
plt.scatter(y_val, val_preds, alpha=0.2, s=5, color='steelblue')
plt.plot([0, 1], [0, 1], 'r--', linewidth=1)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title(f'Actual vs Predicted  (Score: {score:.1f})')
plt.tight_layout()
plt.show()

## 8. Generate Submission

In [ ]:
test_preds = model.predict(X_test)

# Clip to valid range
test_preds = np.clip(test_preds, 0.0, 1.0)

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': test_preds,
})

print(f'Submission shape: {submission.shape}  (expected: 41778 x 2)')
submission.head()

In [ ]:
# Save — edit the filename to reflect your experiment
submission_path = f'../submissions/baseline_rf_score{score:.1f}.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved to: {submission_path}')

---
## What's Next?

| Idea | Expected gain |
|------|---------------|
| Switch to LightGBM / XGBoost | Likely +2–5 pts |
| Geohash → lat/lon decode | Spatial signal |
| Lag features (prev time slot demand) | Strong signal if index ordering is temporal |
| Median demand per geohash | Location baseline |
| Tune `n_estimators`, `max_depth` | Marginal |

**Remember: always validate locally before submitting. Only 50 submissions shared across 4 people.**